# Homework Problem 1

**Date/Time:** 28 Feb 2026 18:22:45 UTC

**Compute:**
- 3rd-body acceleration for sun
- 3rd-body acceleration for moon
- Drag acceleration with F10 = 100  
  *(Remember: model uses F10scaled = F10/100)*
- Solar radiation pressure
- *(For extra fun: Gravitational acceleration for n=m=4)*

**Compute total acceleration on SV**

---

## Homework Inputs

**Vector (Earth-Fixed, Inertial)**

| Parameter | Value          | Units    | Parameter | Value      | Units |
|-----------|----------------|----------|-----------|------------|-------|
| X         | 5907119        | m        | LATC      | 25.82091   | deg   |
| Y         | -1465029       | m        | LATD      | 25.87529   | deg   |
| Z         | 2944866        | m        | LON       | 346.0693   | deg   |
| XD        | 935.1631       | m/sec    | ALTD      | 387.0463   | km    |
| YD        | 7414.308       | m/sec    |           |            |       |
| ZD        | 1901.963       | m/sec    |           |            |       |

- **LATC** = Geocentric latitude  
- **LATD** = Geodetic latitude  
- **LON** = Longitude  
- **ALTD** = Geodetic Altitude  

**Notes:**  
- Vector is inertial, but corresponds to ECEF.  
- No need to do RNP transformation for this case.  
- Pretend that sun/moon vectors are in this frame.

---

## More Inputs (Constants)

- Drag coefficient: \( C_d = 2.0 \)
- Drag Area = \( 10 \, m^2 \)
- Radiation coefficient: \( C_R = 1.41 \)
- Solar Pressure Area = \( 20 \, m^2 \)
- Vehicle mass = \( 1000 \, kg \)
- ω⊕ = 72.921151467 × 10⁻⁶ rad/sec
- 1 nautical mile = 1.852 km (exactly)
- μₘₒₒₙ = μ⊕ / 81.3005764441083
- μₛᵤₙ = μ⊕ · 332946.09358859973

In [4]:
from math import *
from standards import *

#### given info
pos_ = Vector3(5907119, -1465029, 2944866)
vel_ = Vector3(935.1631, 7414.308, 1901.963)
LATC = 25.82091
LATD = 25.87529
LON = 346.0693
ALTD = 387.0463

UTC_year = 2026
UTC_month = 2
UTC_day = 28
UTC_hour = 18
UTC_minute = 22
UTC_second = 45

mu_earth = 3.986004418e14


#### Compute JD_TT
J_date_midnight = (
    floor((1461*(UTC_year+4800+(UTC_month-14)/12))/4)
    +floor((367*(UTC_month-2-12*((UTC_month-14)/12)))/12)
    -floor((3*((UTC_year+4900+(UTC_month-14)/12)/100))/4)
    +UTC_day-32075
)
d = UTC_hour/24+UTC_minute/1440+UTC_second/86400-0.5
J_date = J_date_midnight + d
MJD = J_date - 2400000.5
T = 2000 + (MJD - 51544.03)/365.242199
UT2_UT1 = 0.022*sin(2*pi*T) - 0.012*cos(2*pi*T) - 0.006*sin(4*pi*T) + 0.007*cos(4*pi*T)
TAI_UTC = 37
TT_UTC = TAI_UTC + 32.184
UT1_UTC = 0.0640+0.00003*(MJD-61091) - (UT2_UT1)
Sec_From_J2000 = round((J_date - 2451545.0)*86400)
TAI_Sec = Sec_From_J2000 + TAI_UTC
JD_TT = J_date + TT_UTC/86400


#### compute sun vector 
n = JD_TT-2451545
L = (280.460 + 0.9856474*n)%360
g = (357.528 + 0.9856003*n)%360
Ecliptic_lon = L + 1.915*sin(radians(g))+0.020*sin(2*radians(g))
Ecliptic_lat = 0
Obliqity_of_eliptic = 23.439 - 0.0000004*n
R = 1.00014-0.01671*cos(radians(g))-0.00014*cos(2*radians(g))
x = R*cos(radians(Ecliptic_lon))
y = R*cos(radians(Obliqity_of_eliptic))*sin(radians(Ecliptic_lon))
z = R*sin(radians(Obliqity_of_eliptic))*sin(radians(Ecliptic_lon))
m_in_au = 149597870700
sun_coordinates = Vector3(x*m_in_au, y*m_in_au, z*m_in_au)
sun_ = sun_coordinates.get_np_vector()


#### compute acceleration due to sun
r_rel_ = sun_ - pos_.get_np_vector()
u_sun = mu_earth * 332946.09358859973
a_sun = u_sun * (r_rel_/(np.linalg.norm(r_rel_)**3) - sun_/(np.linalg.norm(sun_)**3))
print("Sun Point Mass Gravity:")
print(a_sun)

Sun Point Mass Gravity:
[[ 4.05357611e-07]
 [-1.54019658e-07]
 [-2.12742334e-07]]


In [5]:
#### compute moon vector:

T = (JD_TT-2451545)/36525

Ecliptic_lon = 218.32+481267.881*T\
+ 6.29*sin(radians(135.0 + 477198.87*T)) - 1.27*sin(radians(259.3 - 413335.36*T))\
+ 0.66*sin(radians(235.7 + 890534.22*T)) + 0.21*sin(radians(269.9 + 954397.74*T))\
- 0.19*sin(radians(357.5 + 35999.05*T)) - 0.11*sin(radians(186.5 + 966404.03*T))

Ecliptic_lat = 5.13*sin(radians(93.3 + 483202.02*T)) + 0.28*sin(radians(228.2 + 960400.89*T))\
- 0.28*sin(radians(318.3 + 6003.15*T)) - 0.17*sin(radians(217.6 - 407332.21*T))

Pi = 0.9508 + 0.0518*cos(radians(135.0 + 477198.87*T)) + 0.0095*cos(radians(259.3 - 413335.36*T))\
+ 0.0078*cos(radians(235.7 + 890534.22*T)) + 0.0028*cos(radians(269.9 + 954397.74*T))
r = 1/sin(radians(Pi))
l = cos(radians(Ecliptic_lat))*cos(radians(Ecliptic_lon))
m = 0.9175*cos(radians(Ecliptic_lat))*sin(radians(Ecliptic_lon)) - 0.3978*sin(radians(Ecliptic_lat))
n = 0.3978*cos(radians(Ecliptic_lat))*sin(radians(Ecliptic_lon)) + 0.9175*sin(radians(Ecliptic_lat))

x = r*l
y = r*m
z = r*n

radius_earth = 6378137 #m

moon_ = Vector3(x*radius_earth, y*radius_earth, z*radius_earth).get_np_vector()

#### compute acceleration due to moon
r_rel_ = moon_ - pos_.get_np_vector()
u_moon = mu_earth / 81.3005764441083
a_moon = u_moon * (r_rel_/(np.linalg.norm(r_rel_)**3) - moon_/(np.linalg.norm(moon_)**3))
print("Moon Point Mass Gravity:")
print(a_moon)

Moon Point Mass Gravity:
[[ 2.53255429e-08]
 [-5.60640510e-07]
 [-6.19984857e-07]]


In [6]:
#### compute drag

w_ = Vector3(0, 0, 72.921151467e-6)
v_r_ = vel_ - (w_.cross(pos_))
cd = 2
A = 10 #m^2
m = 1000 #kg
B = 0.55 #radians
S_ = sun_coordinates.unit_vector()
U_ = Vector3(S_.x*cos(B)-S_.y*sin(B), S_.y*cos(B)+S_.x*sin(B), S_.z)
h = ALTD

#compute days since Dec 31 1957:
UTC_year = 1957
UTC_month = 12
UTC_day = 31
UTC_hour = 0
UTC_minute = 0
UTC_second = 0

mu_earth = 3.986004418e14

J_date_midnight = (
    floor((1461*(UTC_year+4800+(UTC_month-14)/12))/4)
    +floor((367*(UTC_month-2-12*((UTC_month-14)/12)))/12)
    -floor((3*((UTC_year+4900+(UTC_month-14)/12)/100))/4)
    +UTC_day-32075
)
d = UTC_hour/24+UTC_minute/1440+UTC_second/86400-0.5
J_date_1957 = J_date_midnight + d
MJD = J_date_1957 - 2400000.5
T = 2000 + (MJD - 51544.03)/365.242199
UT2_UT1 = 0.022*sin(2*pi*T) - 0.012*cos(2*pi*T) - 0.006*sin(4*pi*T) + 0.007*cos(4*pi*T)
TAI_UTC = 37
TT_UTC = TAI_UTC + 32.184
UT1_UTC = 0.0640+0.00003*(MJD-61091) - (UT2_UT1)
Sec_From_J2000 = round((J_date_1957 - 2451545.0)*86400)
TAI_Sec = Sec_From_J2000 + TAI_UTC
JD_TT_1957 = J_date_1957 + TT_UTC/86400
days_since_1957 = J_date-J_date_1957
#####################################

T = days_since_1957
cos_angle_from_sv_to_diurnal_bulge = pos_.dot(U_) / (pos_.magnitude()*U_.magnitude())
F10_scaled = 1.5+0.8*cos((2*pi*T)/4020)
p_0 = exp((6.363*exp(-0.0048*h)-0.00368*h-15.738)*log(10))
#Nautical altitude is 208.988282937365
p = p_0*(0.85*F10_scaled)*(1+0.02375*(exp(0.0102*h)-1.9)*(1+cos_angle_from_sv_to_diurnal_bulge)**3)*515.37886
print(p)
a_drag = -((cd*A)/(2*m))*p*v_r_.magnitude()**2 * v_r_.unit_vector()
print("Atmospheric Drag:")
print(a_drag.get_np_vector())

3.79624773063938e-13
Atmospheric Drag:
[[-2.29085759e-08]
 [-1.93139195e-07]
 [-5.26012398e-08]]
